# AI for Radiologic Image Tasks: Segmentation

This notebook introduces a complete, educational workflow for training a lung segmentation model on chest X-ray images. The task is to predict a binary mask that marks the lung pixels in each image.

The notebook uses the NLM Montgomery dataset because each chest radiograph is paired with manual masks of the left and right lungs. These masks are combined into a single binary target used to train a segmentation network.

The Montgomery images are split into training, validation, and test groups. Training updates the model weights, validation selects the checkpoint, and the held-out test split gives a final estimate after model selection.

The main steps are:

1. Build an image index with image paths, mask paths, and readable disease labels.
2. Load chest X-ray images and combine left-lung and right-lung masks.
3. Create a stratified train/validation/test split from the annotated Montgomery images.
4. Prepare datasets and data loaders, including mask-aware image augmentation.
5. Define an encoder-decoder network for pixel-level lung segmentation.
6. Train the model while monitoring loss, overlap, sensitivity, specificity, and pixel-accuracy metrics on the validation split.
7. Evaluate and preview model predictions on the held-out test split.

### Task at a glance

`Chest X-ray + lung mask → Prepare aligned pairs → Train and validate → Predicted lung mask → Test → Inspect errors`

> **For beginners:** collapsed 🔒 cells contain setup or implementation details and can be run without editing. Visible ✏️ control panels contain the safest settings to experiment with. Definitions of technical terms and solutions to common errors are available in the project README.

The goal is not to build a production-ready medical segmentation system. Instead, the notebook is meant to show the core logic behind segmentation: pair each image with a pixel-level target, keep images and masks aligned during preprocessing, train a model to predict one value per pixel, and evaluate whether the predicted anatomy overlaps the manual annotation.

> **Google Colab quick start:** open this notebook in Colab, select **Runtime → Change runtime type → T4 GPU** (recommended for training), then choose **Runtime → Run all**. The setup cell installs missing libraries and downloads the public dataset automatically; no manual uploads or path edits are needed.


## 0. One-click setup

Run the following two collapsed cells exactly as they are. The first installs missing libraries and downloads the public dataset, while the second imports the libraries and configures the runtime. In Colab, choose a GPU before running all cells for a much faster training run.

The downloaded data and model results live only for this Colab session unless you explicitly save them elsewhere.


In [ ]:
#@title Setup — install dependencies and download data
import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

def install(packages):
    """Install the small set of extra libraries used by this notebook."""
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *packages])

DATASET_URL = "https://huggingface.co/datasets/Famatsu123/montgomery-shenzhen-tuberculosis-cxr/resolve/main"
DATA_ROOT = (Path("/content") if IN_COLAB else Path.cwd()) / "datasets"

def get_dataset(archive_name, folder):
    """Download one public archive and extract it once; later runs reuse it."""
    if folder.exists():
        return folder
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    archive = DATA_ROOT / archive_name
    if not archive.exists():
        print(f"Downloading {archive_name} from Hugging Face (first run only)...")
        urllib.request.urlretrieve(f"{DATASET_URL}/{archive_name}?download=true", archive)
    print(f"Extracting {archive_name}...")
    with zipfile.ZipFile(archive) as zip_file:
        zip_file.extractall(DATA_ROOT)
    return folder

PACKAGES = ["albumentations", "opencv-python-headless"]
install(PACKAGES)

DATASET_DIR = get_dataset("MontgomerySet.zip", DATA_ROOT / "MontgomerySet")
IMAGE_DIR = DATASET_DIR / "CXR_png"
LEFT_MASK_DIR = DATASET_DIR / "ManualMask" / "leftMask"
RIGHT_MASK_DIR = DATASET_DIR / "ManualMask" / "rightMask"

In [ ]:
#@title Imports and runtime configuration
from collections import OrderedDict
import random
import warnings

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams["figure.figsize"] = (8, 8)
plt.rcParams["image.cmap"] = "gray"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

assert IMAGE_DIR.exists(), f"Image directory not found: {IMAGE_DIR}"

assert LEFT_MASK_DIR.exists(), f"Left masks not found: {LEFT_MASK_DIR}"
assert RIGHT_MASK_DIR.exists(), f"Right masks not found: {RIGHT_MASK_DIR}"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {DEVICE}")
print("Dataset is ready. Re-running this cell reuses the downloaded files.")


## 1. Build Image And Mask Index

Here we create a simple table with one row per image. Each row stores the image ID, image path, left-mask path, right-mask path, numeric disease label, and readable label name. This table becomes the source of truth for both the image pixels and the manual segmentation masks used later in the notebook.

The disease label is not used as the segmentation target. It is kept so the train/validation/test split can preserve a similar TB / No TB proportion in each group, matching the audience-facing split logic used in the classification and detection notebooks.

**What are we doing?** Matching each radiograph to its left- and right-lung masks. **Why?** Segmentation requires exact image-mask pairs. **Expected result:** one complete record per annotated image. **⚠️ Watch for:** the TB label balances the split; lung/background pixels are the actual segmentation targets.


In [ ]:
#@title 🔒 Implementation — build the image and mask index
def disease_label_from_id(image_id: str) -> int:
    return int(image_id.rsplit("_", 1)[-1])


def build_montgomery_index() -> pd.DataFrame:
    rows = []
    for image_path in sorted(IMAGE_DIR.glob("*.png")):
        left_mask_path = LEFT_MASK_DIR / image_path.name
        right_mask_path = RIGHT_MASK_DIR / image_path.name
        if not left_mask_path.exists() or not right_mask_path.exists():
            continue
        image_id = image_path.stem
        rows.append({
            "ImageId": image_id,
            "image_path": image_path,
            "left_mask_path": left_mask_path,
            "right_mask_path": right_mask_path,
            "disease_label": disease_label_from_id(image_id),
        })
    return pd.DataFrame(rows)


images_df = build_montgomery_index()
assert not images_df.empty, "No complete image/left-mask/right-mask triplets found."
print(f"paired images: {len(images_df):,}")
images_df["disease_label"].value_counts().rename({0: "normal", 1: "abnormal"})


### 👀 How to read the index result

The total and label counts confirm which source files are available. They describe the dataset before splitting or training and should not be interpreted as model performance.

## 2. Image And Mask Loading

This section reads Montgomery chest X-ray images and their manual lung masks. The image files are loaded as grayscale arrays, and the left-lung and right-lung masks are combined into one binary mask.

A binary mask stores one target value per pixel: background pixels are 0, and lung pixels are 1. This makes segmentation more detailed than classification or box detection because the model must learn the approximate shape and boundary of the lung fields.

**What are we doing?** Reading the radiograph and combining the two lung masks into one binary target. **Why?** The model learns one `lung` class rather than separate left/right classes. **Expected result:** an image and mask with identical height and width. **⚠️ Watch for:** any spatial mismatch would teach the model an incorrect boundary.


In [ ]:
#@title 🔒 Implementation — load images and masks
def read_grayscale_png(path: Path) -> np.ndarray:
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise FileNotFoundError(path)
    return image


def normalize_percentile(image: np.ndarray, lower: float = 1, upper: float = 99, eps: float = 1e-7) -> np.ndarray:
    image = image.astype(np.float32)
    lo, hi = np.percentile(image, [lower, upper])
    image = np.clip(image, lo, hi)
    return ((image - lo) / (hi - lo + eps)).astype(np.float32)


def lung_mask(row: pd.Series) -> np.ndarray:
    left = read_grayscale_png(row.left_mask_path) > 0
    right = read_grayscale_png(row.right_mask_path) > 0
    return (left | right).astype(np.uint8)


def image_and_mask(image_id: str) -> tuple[np.ndarray, np.ndarray]:
    row = images_df.loc[images_df["ImageId"] == image_id].iloc[0]
    image = read_grayscale_png(row.image_path)
    mask = lung_mask(row)
    return image, mask


The next cell displays one Montgomery image, its combined lung mask, and an overlay of that mask on the image. This is a quick sanity check that the manual masks are aligned with the visible lung fields before model training starts. The plain image shows the input, the binary mask shows the target, and the overlay makes correspondence easier to judge.


In [ ]:
sample_row = images_df.sample(1, random_state=SEED).iloc[0]
sample_image = read_grayscale_png(sample_row.image_path)
sample_mask = lung_mask(sample_row)

print(f"sample ImageId: {sample_row.ImageId}")
print(f"image shape: {sample_image.shape}")
print(f"mask positive pixels: {int(sample_mask.sum()):,}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(normalize_percentile(sample_image), cmap="gray")
axes[0].set_title("Image")
axes[1].imshow(sample_mask, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Combined lung mask")
axes[2].imshow(normalize_percentile(sample_image), cmap="gray")
axes[2].imshow(np.ma.masked_where(sample_mask == 0, sample_mask), alpha=0.45, cmap="Reds", vmin=0, vmax=1)
axes[2].set_title("Image + mask")
for ax in axes:
    ax.axis("off")
plt.tight_layout()


### 👀 How to read the image–mask example

The mask should cover the visible left and right lung fields, and its shape should match the printed image shape. The positive-pixel count is an annotation-size check, not a quality score. Inspect for masks shifted away from the anatomy or empty masks.

## 3. Train/Validation/Test Split

The annotated Montgomery dataset is divided into three groups before training:

- **Training split:** 70% of the images, used to update the model weights.
- **Validation split:** 15% of the images, kept separate and used to monitor performance during training and select the best checkpoint.
- **Test split:** 15% of the images, held out until the final evaluation.

The split is **stratified by disease label**, meaning the notebook preserves a similar TB / No TB proportion in each group. The model is learning lung boundaries rather than TB classification, but stratification still helps avoid putting most abnormal examples in only one split.

The validation split guides checkpoint selection. The test split is used only after training, so it gives a cleaner final estimate of how well the selected segmenter overlaps manual masks on held-out images from the same annotated source.

**What are we doing?** Creating non-overlapping image groups. **Why?** Reusing training images for final evaluation would exaggerate performance. **Expected result:** about 70%/15%/15% of images with similar abnormal fractions. **⚠️ Watch for:** both masks belonging to one radiograph must remain with that radiograph in a single split.


In [ ]:
#@title 🔒 Implementation — create stratified splits
def stratified_image_split(
    df: pd.DataFrame,
    valid_fraction: float = 0.15,
    test_fraction: float = 0.15,
    seed: int = SEED,
):
    if valid_fraction <= 0 or test_fraction <= 0 or valid_fraction + test_fraction >= 1:
        raise ValueError("valid_fraction and test_fraction must be positive and sum to less than 1.")

    rng = np.random.default_rng(seed)
    train_ids, valid_ids, test_ids = [], [], []
    for _, group in df.groupby("disease_label"):
        ids = group["ImageId"].to_numpy().copy()
        rng.shuffle(ids)
        n_valid = max(1, int(round(len(ids) * valid_fraction)))
        n_test = max(1, int(round(len(ids) * test_fraction)))
        if n_valid + n_test >= len(ids):
            raise ValueError("Not enough images in each class to create train, validation, and test splits.")
        valid_ids.extend(ids[:n_valid])
        test_ids.extend(ids[n_valid:n_valid + n_test])
        train_ids.extend(ids[n_valid + n_test:])

    train_ids = np.array(train_ids)
    valid_ids = np.array(valid_ids)
    test_ids = np.array(test_ids)
    rng.shuffle(train_ids)
    rng.shuffle(valid_ids)
    rng.shuffle(test_ids)
    return train_ids, valid_ids, test_ids


train_ids, valid_ids, test_ids = stratified_image_split(images_df)
label_lookup = images_df.set_index("ImageId")["disease_label"]
pd.DataFrame({
    "split": ["all", "train", "valid", "test"],
    "images": [len(images_df), len(train_ids), len(valid_ids), len(test_ids)],
    "abnormal_fraction": [
        images_df["disease_label"].mean(),
        label_lookup.loc[train_ids].mean(),
        label_lookup.loc[valid_ids].mean(),
        label_lookup.loc[test_ids].mean(),
    ],
})


### 👀 How to read the split table

The `images` column reports group size. Similar `abnormal_fraction` values show that normal and abnormal studies are distributed comparably, even though disease status is not the pixel target.

## 4. Image and Mask Preparation

This section prepares images and masks for the model by defining the transformations and the custom PyTorch dataset class.

Data augmentation is more delicate for segmentation than for classification. If an image is shifted, scaled, rotated, or flipped, the mask must be transformed in the same way. Otherwise, the pixels marked as lung would no longer match the lung pixels in the image.

In this notebook, the training images use these augmentation techniques:

1. Resize: makes every image and mask the same input size for the neural network.
2. Shift, scale, and rotation: slightly moves, zooms, or rotates the image and mask together.
3. Horizontal flip: mirrors the image and mask together.

Validation and test images are not randomly augmented. They are only resized, so validation and test loss, Dice score, IoU, sensitivity, specificity, and pixel accuracy are consistent and easier to interpret.

**What are we doing?** Converting paired images and masks into aligned tensors and defining joint augmentation. **Why?** The network needs uniform dimensions and realistic variation. **Expected result:** image and mask batches with matching batch size, height, and width. **⚠️ Watch for:** applying a geometric transformation to the image alone destroys the target correspondence.


In [ ]:
#@title 🔒 Implementation — transforms and dataset class
def make_train_transform(size: int) -> A.Compose:
    return A.Compose([
        A.Resize(height=size, width=size),
        A.ShiftScaleRotate(
            shift_limit=0.03,
            scale_limit=0.08,
            rotate_limit=7,
            p=0.5,
            border_mode=cv2.BORDER_CONSTANT,
        ),
        A.HorizontalFlip(p=0.5),
    ])


def make_valid_transform(size: int) -> A.Compose:
    return A.Compose([
        A.Resize(height=size, width=size),
    ])


class MontgomeryLungDataset(Dataset):
    def __init__(self, image_ids, metadata: pd.DataFrame, transform: A.Compose | None = None):
        self.image_ids = list(image_ids)
        self.metadata = metadata.set_index("ImageId")
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        row = self.metadata.loc[image_id]
        image = normalize_percentile(read_grayscale_png(row.image_path))
        mask = lung_mask(row).astype(np.float32)

        if self.transform is not None:
            transformed = self.transform(image=image, mask=mask)
            image = transformed["image"].astype(np.float32)
            mask = transformed["mask"].astype(np.float32)

        image = np.repeat(image[..., None], 3, axis=-1)
        image = (image - IMAGENET_MEAN) / IMAGENET_STD

        image = np.ascontiguousarray(image.transpose(2, 0, 1))
        mask = np.ascontiguousarray((mask > 0.5).astype(np.float32)[None, ...])

        image = torch.from_numpy(image)
        mask = torch.from_numpy(mask)
        return image, mask, image_id


In the next cell, we use the dataset class defined above and choose the main data-loading settings for this project.

`IMAGE_SIZE` defines the standard image size used by the model. `BATCH_SIZE` controls how many image-mask pairs are processed at once. `NUM_WORKERS` controls how many background processes help load the data.

> ✏️ **Control panel:** reduce `BATCH_SIZE` if memory is limited. The optional `MAX_*_IMAGES` settings support quick smoke tests, but metrics from a small subset should not be treated as meaningful performance estimates.


In [ ]:
#@title ✏️ Control panel — data settings
IMAGE_SIZE = 512
BATCH_SIZE = 8
NUM_WORKERS = 2 if IN_COLAB else 4

# Keep these non-None for quick CPU smoke tests. Set to None for full training.
MAX_TRAIN_IMAGES = None
MAX_VALID_IMAGES = None
MAX_TEST_IMAGES = None


In [ ]:
#@title 🔒 Implementation — create datasets and data loaders
train_transform = make_train_transform(IMAGE_SIZE)
valid_transform = make_valid_transform(IMAGE_SIZE)

train_ids_run = train_ids[:MAX_TRAIN_IMAGES] if MAX_TRAIN_IMAGES else train_ids
valid_ids_run = valid_ids[:MAX_VALID_IMAGES] if MAX_VALID_IMAGES else valid_ids
test_ids_run = test_ids[:MAX_TEST_IMAGES] if MAX_TEST_IMAGES else test_ids

train_ds = MontgomeryLungDataset(train_ids_run, images_df, transform=train_transform)
valid_ds = MontgomeryLungDataset(valid_ids_run, images_df, transform=valid_transform)
test_ds = MontgomeryLungDataset(test_ids_run, images_df, transform=valid_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

images, masks, image_ids = next(iter(train_loader))
print(f"Image batch shape: {list(images.shape)} = {images.shape[0]} images, {images.shape[1]} channels, {images.shape[2]} × {images.shape[3]} pixels")
print(f"Mask batch shape: {list(masks.shape)} = {masks.shape[0]} masks, {masks.shape[1]} binary channel, {masks.shape[2]} × {masks.shape[3]} pixels")
print(f"The matching batch size and height/width show that every image remains aligned with its mask.")
print(f"Example image IDs: {list(image_ids[:2])}")
print(f"Validation images: {len(valid_ds)}; held-out test images: {len(test_ds)}")


### 👀 How to read the tensor shapes

An image batch such as `[8, 3, 512, 512]` means 8 images, 3 channels, and 512 × 512 pixels. Its mask batch should be `[8, 1, 512, 512]`: one binary target channel per matching image. Agreement in batch size and spatial dimensions confirms alignment.

The next cell shows how mask-aware augmentation changes a training example. The first column displays the resized image and mask without random augmentation, and the other columns show augmented versions created from the same original image with the mask transformed alongside it.


In [ ]:
preview_id = sample_row.ImageId
preview_image, preview_mask = image_and_mask(preview_id)
preview_image = normalize_percentile(preview_image).astype(np.float32)
preview_mask = preview_mask.astype(np.float32)

base = valid_transform(image=preview_image, mask=preview_mask)
base_image, base_mask = base["image"], base["mask"]

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for col in range(4):
    if col == 0:
        image_aug, mask_aug = base_image, base_mask
        title = "Original"
    else:
        augmented = train_transform(image=preview_image, mask=preview_mask)
        image_aug, mask_aug = augmented["image"], augmented["mask"]
        title = f"Albumentations {col}"

    axes[0, col].imshow(image_aug, cmap="gray", vmin=0, vmax=1)
    axes[0, col].set_title(title)
    axes[1, col].imshow(image_aug, cmap="gray", vmin=0, vmax=1)
    axes[1, col].imshow(np.ma.masked_where(mask_aug == 0, mask_aug), alpha=0.45, cmap="Reds", vmin=0, vmax=1)
    axes[1, col].set_title("Mask overlay")

for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()


### 👀 How to read the augmentation preview

The lower-row overlay should remain aligned with the lungs in every column. The changes should be modest and anatomically plausible. Misalignment here would invalidate training even if later code ran successfully.

## 5. Image-Mask Correspondence

Segmentation changes the annotation from one value per image to one value per pixel. Each pixel is assigned either to the target anatomy or to the background.

### The mask is part of the training data.

A mask is spatially tied to the image. If the image is resized, the mask must be resized. If the image is flipped, the mask must be flipped. If the image is rotated, the mask must rotate with it. The model only learns a useful anatomy boundary when those two arrays stay aligned.

This is why the transform pipeline receives both `image` and `mask`, and why validation uses deterministic preprocessing. The training examples can vary, but the relationship between the radiograph and its lung mask must remain correct.

**Takeaway:** segmentation is dense supervision. The model is not only deciding whether something is present; it is learning where each relevant pixel belongs.

### Pixel agreement by picture

| Pixel color in the final preview | Meaning | Metric role |
| :--- | :--- | :--- |
| Yellow | Reference and prediction agree on lung | True positive / overlap |
| Cyan | Reference says lung, prediction misses it | False negative |
| Red | Prediction says lung outside the reference | False positive |
| Black | Both agree on background | True negative |

`IoU = overlap ÷ combined lung area`

`Dice = 2 × overlap ÷ (reference lung area + predicted lung area)`

Both scores range from 0 to 1 and are better when higher. Dice gives the shared area twice the weight and is therefore numerically higher than IoU for the same pair of non-perfect masks.


## 6. Segmentation Model

This section defines an encoder-decoder segmentation network. The encoder extracts increasingly abstract image features, whereas the decoder restores spatial resolution to produce a dense prediction map.

The model produces one prediction for each pixel. During training, these predictions are compared with the binary reference mask; during visualization, they are converted to a predicted mask.

This architecture is useful for teaching segmentation because it shows the key idea clearly: combine high-level context from deep layers with spatial detail from earlier layers to recover a dense pixel-level prediction.

**What are we doing?** Building an encoder-decoder model with a pretrained ResNet34 encoder. **Why?** The encoder recognizes image patterns while the decoder restores their spatial location. **Expected result:** one output logit for every input pixel. **⚠️ Watch for:** an output with the wrong height or width cannot be compared directly with the reference mask.

> ✏️ **Control panel:** disabling `PRETRAINED_ENCODER` avoids downloading weights but usually makes learning from this small dataset harder.


In [ ]:
#@title 🔒 Implementation — segmentation model architecture
RESNET34_URL = "https://download.pytorch.org/models/resnet34-b627a593.pth"


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes: int, planes: int, stride: int = 1, downsample: nn.Module | None = None):
        super().__init__()
        self.conv1 = nn.Conv2d(inplanes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        return self.relu(out)


class ResNet34Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.inplanes = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(64, 3)
        self.layer2 = self._make_layer(128, 4, stride=2)
        self.layer3 = self._make_layer(256, 6, stride=2)
        self.layer4 = self._make_layer(512, 3, stride=2)

    def _make_layer(self, planes: int, blocks: int, stride: int = 1):
        downsample = None
        if stride != 1 or self.inplanes != planes:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes),
            )
        layers = [BasicBlock(self.inplanes, planes, stride, downsample)]
        self.inplanes = planes
        layers.extend(BasicBlock(self.inplanes, planes) for _ in range(1, blocks))
        return nn.Sequential(*layers)

    def load_imagenet_weights(self):
        state_dict = torch.hub.load_state_dict_from_url(RESNET34_URL, map_location="cpu", progress=True)
        encoder_state = OrderedDict((key, value) for key, value in state_dict.items() if not key.startswith("fc."))
        self.load_state_dict(encoder_state, strict=True)

    def forward(self, x):
        x0 = x
        x1 = self.relu(self.bn1(self.conv1(x)))
        x2 = self.maxpool(x1)
        x2 = self.layer1(x2)
        x3 = self.layer2(x2)
        x4 = self.layer3(x3)
        x5 = self.layer4(x4)
        return x0, x1, x2, x3, x4, x5


class DecoderBlock(nn.Module):
    def __init__(self, in_channels: int, skip_channels: int, out_channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels + skip_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.block(x)


class ResNet34UNet(nn.Module):
    def __init__(self, out_channels: int = 1, pretrained: bool = True):
        super().__init__()
        self.encoder = ResNet34Encoder()
        if pretrained:
            self.encoder.load_imagenet_weights()
        self.decoder4 = DecoderBlock(512, 256, 256)
        self.decoder3 = DecoderBlock(256, 128, 128)
        self.decoder2 = DecoderBlock(128, 64, 64)
        self.decoder1 = DecoderBlock(64, 64, 32)
        self.decoder0 = DecoderBlock(32, 3, 32)
        self.final_conv = nn.Conv2d(32, out_channels, kernel_size=1)

    def forward(self, x):
        x0, x1, x2, x3, x4, x5 = self.encoder(x)
        x = self.decoder4(x5, x4)
        x = self.decoder3(x, x3)
        x = self.decoder2(x, x2)
        x = self.decoder1(x, x1)
        x = self.decoder0(x, x0)
        return self.final_conv(x)


The next cell creates the model and verifies that its output has the expected spatial dimensions. A shape such as `[2, 1, 512, 512]` means two predicted masks, one output channel, and one logit for each 512 × 512 pixel location. This confirms shape compatibility, not segmentation quality.


In [ ]:
#@title ✏️ Control panel — model setting
PRETRAINED_ENCODER = True

model = ResNet34UNet(pretrained=PRETRAINED_ENCODER).to(DEVICE)
with torch.no_grad():
    test_logits = model(images[:2].to(DEVICE))
test_logits.shape


### 👀 How to read the model check

The output shape should match the reference-mask shape: one logit channel for every input pixel. This verifies that data can pass through the network; it does not show that the untrained segmentation is anatomically correct.

## 7. Train

Training repeatedly shows image-and-mask batches to the model, measures prediction error, and updates the model weights. The notebook tracks six quantities:

- **Segmentation loss:** a combination of binary cross-entropy and Dice loss used for optimization.
- **Dice score:** an overlap score that emphasizes agreement between predicted and target lung pixels.
- **Intersection-over-union (IoU):** another overlap score between the predicted mask and the ground-truth mask.
- **Sensitivity:** the proportion of reference lung pixels recovered by the prediction.
- **Specificity:** the proportion of background pixels correctly classified as background.
- **Pixel accuracy:** the proportion of all pixels classified correctly.

Higher Dice, IoU, sensitivity, specificity, and pixel accuracy values indicate better agreement with the manual masks. The best checkpoint is selected by validation loss. Because background pixels substantially outnumber lung pixels, Dice and IoU are generally more informative overlap measures than pixel accuracy and should be complemented by visual review.

**What are we doing?** Updating model weights on training masks and checking validation masks after every epoch. **Why?** Validation behavior selects the checkpoint without using test data. **Expected result:** loss generally decreases while Dice and IoU increase before leveling off. **⚠️ Watch for:** high pixel accuracy can be misleading because most pixels are background.


In [ ]:
#@title 🔒 Implementation — segmentation metrics and loss
def segmentation_scores_from_logits(logits: torch.Tensor, targets: torch.Tensor, threshold: float = 0.5, eps: float = 1e-7) -> dict[str, torch.Tensor]:
    preds = (torch.sigmoid(logits) > threshold).float()
    dims = tuple(range(1, preds.ndim))
    intersection = (preds * targets).sum(dim=dims)
    pred_area = preds.sum(dim=dims)
    target_area = targets.sum(dim=dims)
    union = pred_area + target_area - intersection
    true_negative = ((1 - preds) * (1 - targets)).sum(dim=dims)
    false_positive = (preds * (1 - targets)).sum(dim=dims)
    false_negative = ((1 - preds) * targets).sum(dim=dims)

    dice_per_image = (2 * intersection + eps) / (pred_area + target_area + eps)
    iou_per_image = (intersection + eps) / (union + eps)
    sensitivity_per_image = (intersection + eps) / (intersection + false_negative + eps)
    specificity_per_image = (true_negative + eps) / (true_negative + false_positive + eps)
    pixel_accuracy_per_image = (intersection + true_negative + eps) / (intersection + true_negative + false_positive + false_negative + eps)

    return {
        "dice": dice_per_image.mean(),
        "iou": iou_per_image.mean(),
        "sensitivity": sensitivity_per_image.mean(),
        "specificity": specificity_per_image.mean(),
        "pixel_accuracy": pixel_accuracy_per_image.mean(),
    }


def dice_loss_from_logits(logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-7) -> torch.Tensor:
    probs = torch.sigmoid(logits)
    dims = tuple(range(1, probs.ndim))
    intersection = (probs * targets).sum(dim=dims)
    denominator = probs.sum(dim=dims) + targets.sum(dim=dims)
    dice = (2 * intersection + eps) / (denominator + eps)
    return 1 - dice.mean()


bce_loss = nn.BCEWithLogitsLoss()


def segmentation_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    return bce_loss(logits, targets) + dice_loss_from_logits(logits, targets)


The next collapsed cell defines the shared epoch loop used for both training and validation. When an optimizer is provided, the model updates its weights. Without an optimizer, the same loop runs in evaluation mode and only records metrics.


In [ ]:
#@title 🔒 Implementation — shared training and validation pass
def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    totals = {"loss": 0.0, "dice": 0.0, "iou": 0.0, "sensitivity": 0.0, "specificity": 0.0, "pixel_accuracy": 0.0}
    total_batches = 0

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for images, masks, _ in tqdm(loader, leave=False):
            images = images.to(DEVICE, non_blocking=True)
            masks = masks.to(DEVICE, non_blocking=True)

            logits = model(images)
            loss = segmentation_loss(logits, masks)

            if is_train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

            scores = segmentation_scores_from_logits(logits, masks)
            totals["loss"] += loss.item()
            for key, value in scores.items():
                totals[key] += value.item()
            total_batches += 1

    return {key: value / total_batches for key, value in totals.items()}


The visible control panel sets the number of epochs and learning rates. The following collapsed cell performs training, saves the best checkpoint by validation loss, and prints a compact progress summary. The test split is not used for checkpoint selection.

> ✏️ **Control panel:** more epochs give the model more opportunities to learn but increase runtime and can increase overfitting. Change one setting at a time and compare the validation trend.


In [ ]:
#@title ✏️ Control panel — training settings
EPOCHS = 10
LEARNING_RATE = 1e-3
MIN_LEARNING_RATE = 1e-5


In [ ]:
#@title 🔒 Implementation — train and save the best checkpoint
CHECKPOINT_PATH = RESULTS_DIR / "resnet34_unet_montgomery_lung.pt"

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=MIN_LEARNING_RATE)
history = []
best_valid_loss = float("inf")
best_model_state = None

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(model, train_loader, optimizer)
    valid_metrics = run_epoch(model, valid_loader)
    current_lr = optimizer.param_groups[0]["lr"]
    history.append({"epoch": epoch, "lr": current_lr, **{f"train_{k}": v for k, v in train_metrics.items()}, **{f"valid_{k}": v for k, v in valid_metrics.items()}})

    if valid_metrics["loss"] < best_valid_loss:
        best_valid_loss = valid_metrics["loss"]
        best_model_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "history": history,
            "image_size": IMAGE_SIZE,
            "train_dataset": "Montgomery",
            "validation_dataset": "Montgomery",
            "test_dataset": "Montgomery",
            "split_seed": SEED,
        }, CHECKPOINT_PATH)

    scheduler.step()

    print(
        f"epoch {epoch:02d} lr {current_lr:.2e} | "
        f"train loss {train_metrics['loss']:.4f} dice {train_metrics['dice']:.4f} iou {train_metrics['iou']:.4f} | "
        f"valid loss {valid_metrics['loss']:.4f} dice {valid_metrics['dice']:.4f} iou {valid_metrics['iou']:.4f}"
    )

if best_model_state is not None:
    model.load_state_dict(best_model_state)

print(f"Best model checkpoint saved to: {CHECKPOINT_PATH}")
pd.DataFrame(history)


### 👀 How to read the training log

Each row compares training and validation loss, Dice, and IoU for one epoch. Lower loss and higher overlap scores are favorable. If training continues improving while validation stalls or worsens, the model may be overfitting; the saved validation-selected checkpoint helps avoid simply using the final epoch.

## 8. Test And Preview Predictions

After training, the best validation checkpoint is evaluated on the held-out test split. The preview shows a 2x2 grid of test-set mask comparisons. Each panel contains only masks: the ground-truth mask in one color and the predicted mask in another color.

The held-out report includes Dice coefficient, IoU, sensitivity, specificity, and pixel accuracy. Dice and IoU summarize foreground overlap; sensitivity measures how much reference lung tissue is recovered, specificity measures correct background classification, and pixel accuracy summarizes all correctly classified pixels. Because background pixels substantially outnumber lung pixels, pixel accuracy can appear strong even when a mask is imperfect, so it should be interpreted alongside Dice, IoU, and visual review.

This visual check is important because aggregate metrics do not show the full failure mode. A model can have a reasonable average score while still missing an apex, leaking outside the thorax, or producing rough boundaries on individual images. Looking at examples helps connect the metric back to the anatomy.

**⚠️ Watch for:** the default probability threshold is 0.5. Changing it alters which pixels become lung but does not retrain the model. A few displayed masks cannot replace full-set metrics.


In [ ]:
#@title 🔒 Implementation — test metrics and mask previews
def predict_batch(model, images: torch.Tensor, threshold: float = 0.5) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(images.to(DEVICE))).cpu().numpy()
    return (probs > threshold).astype(np.uint8), probs


test_metrics = run_epoch(model, test_loader)
print("Held-out test metrics:")
display(pd.DataFrame([{f"test_{key}": value for key, value in test_metrics.items()}]))

preview_test_ids = test_ids_run[:max(BATCH_SIZE, 4)]
preview_test_ds = MontgomeryLungDataset(preview_test_ids, images_df, transform=valid_transform)
preview_test_loader = DataLoader(
    preview_test_ds,
    batch_size=min(BATCH_SIZE, len(preview_test_ds)),
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

images, masks, image_ids = next(iter(preview_test_loader))
pred_masks, pred_probs = predict_batch(model, images)

n = min(4, images.shape[0])
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes = axes.reshape(-1)

for i in range(n):
    ground_truth = masks[i, 0].numpy() > 0.5
    prediction = pred_masks[i, 0] > 0
    mask_overlay = np.zeros((*ground_truth.shape, 3), dtype=np.float32)
    mask_overlay[ground_truth] = np.array([0.0, 0.75, 1.0], dtype=np.float32)
    mask_overlay[prediction] = np.array([1.0, 0.25, 0.15], dtype=np.float32)
    mask_overlay[ground_truth & prediction] = np.array([0.95, 0.85, 0.1], dtype=np.float32)

    axes[i].imshow(mask_overlay, vmin=0, vmax=1)
    axes[i].set_title(f"{image_ids[i]}\nGT cyan | Pred red | Overlap yellow")
    axes[i].axis("off")

for ax in axes[n:]:
    ax.axis("off")
plt.tight_layout()


### 👀 How to read the held-out result

Higher held-out Dice, IoU, sensitivity, specificity, and pixel accuracy are favorable, while lower test loss is favorable. In the overlays, yellow is correct lung overlap, cyan is missed reference lung, red is extra predicted region, and black is agreed background. Inspect apices, costophrenic angles, and leakage beyond the thorax rather than looking only at the average score.

The final cell plots the training history. The loss curve shows whether optimization improved over time; Dice, IoU, sensitivity, specificity, and pixel-accuracy curves summarize segmentation performance; and the learning-rate plot shows the cosine schedule used during training. Test metrics are reported once in the previous cell rather than being part of the training history.


In [ ]:
history_df = pd.DataFrame(history)
history_df.plot(x="epoch", y=["train_loss", "valid_loss", "train_dice", "valid_dice", "train_iou", "valid_iou", "train_sensitivity", "valid_sensitivity", "train_specificity", "valid_specificity", "train_pixel_accuracy", "valid_pixel_accuracy"], figsize=(10, 5))
plt.grid(True, alpha=0.3)
plt.tight_layout()

history_df.plot(x="epoch", y="lr", figsize=(8, 2), logy=True, legend=False)
plt.ylabel("learning rate")
plt.grid(True, alpha=0.3)
plt.tight_layout()


### 👀 How to read the training curves

Loss curves are generally better when they fall; overlap curves are better when they rise. A persistent gap between training and validation suggests weaker generalization. The learning-rate curve shows the planned update-size schedule rather than model quality.

## 🧪 Try it yourself

- **Mask threshold:** compare predictions at `0.3`, `0.5`, and `0.7`. Lower thresholds usually include more lung and more leakage; higher thresholds usually produce smaller masks and may miss boundaries. No retraining is required.
- **More or fewer epochs:** change `EPOCHS`. More epochs increase runtime and may improve training scores without improving validation.
- **Augmentation:** set one transformation probability `p` to `0`, then rerun from image preparation. First confirm in the preview that image and mask remain aligned.
- **Quick smoke test:** set `MAX_TRAIN_IMAGES`, `MAX_VALID_IMAGES`, and `MAX_TEST_IMAGES` to small values. Use this only to verify execution, not to estimate performance.

**Takeaway:** segmentation predicts a dense lung mask. Overlap metrics and qualitative boundary review provide complementary information, and neither establishes clinical validity on its own.
